# VisionGuard — Fine-tuning YOLOv11 on a PPE Dataset

This notebook trains a custom YOLOv11 model to detect PPE violations (missing helmets, missing vests) on construction sites.

**Runtime:** GPU (T4 recommended, free on Colab)

**Dataset:** We use a PPE dataset from Roboflow Universe. You can swap it for your own labeled data.

## 1. Install dependencies

In [ ]:
!pip install ultralytics roboflow -q

## 2. Download dataset from Roboflow

In [ ]:
from roboflow import Roboflow

# Replace with your Roboflow API key and project
# Free account at roboflow.com gives you 10k images/month
rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")
project = rf.workspace("roboflow-universe-projects").project("construction-site-safety")
dataset = project.version(1).download("yolov11")

print(f"Dataset location: {dataset.location}")
print("Classes:", dataset.classes)

## 3. Inspect the dataset

In [ ]:
import os
from pathlib import Path

data_yaml = Path(dataset.location) / "data.yaml"

with open(data_yaml) as f:
    print(f.read())

# Count images
for split in ["train", "valid", "test"]:
    images_dir = Path(dataset.location) / split / "images"
    if images_dir.exists():
        count = len(list(images_dir.glob("*.jpg")) + list(images_dir.glob("*.png")))
        print(f"{split}: {count} images")

## 4. Visualize sample annotations

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
import random

train_images = list((Path(dataset.location) / "train" / "images").glob("*.jpg"))
sample_paths = random.sample(train_images, min(4, len(train_images)))

fig, axes = plt.subplots(1, len(sample_paths), figsize=(16, 4))
for ax, path in zip(axes, sample_paths):
    ax.imshow(mpimg.imread(path))
    ax.set_title(path.name[:20])
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Fine-tune YOLOv11n

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv11 nano — starts from COCO weights
model = YOLO("yolo11n.pt")

results = model.train(
    data=str(data_yaml),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,               # GPU
    patience=10,            # early stopping
    save=True,
    project="runs/ppe",
    name="visionguard_v1",
    # Augmentations — help generalize to different lighting/angles
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
)

## 6. Evaluate on test set

In [ ]:
# Load the best checkpoint
best_model = YOLO("runs/ppe/visionguard_v1/weights/best.pt")

metrics = best_model.val(data=str(data_yaml), split="test")
print(f"mAP@0.5:    {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision:  {metrics.box.mp:.4f}")
print(f"Recall:     {metrics.box.mr:.4f}")

## 7. Run inference on a sample image

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

test_images = list((Path(dataset.location) / "test" / "images").glob("*.jpg"))
sample_img  = str(random.choice(test_images))

results = best_model.predict(sample_img, conf=0.25, save=False)
annotated = results[0].plot()

plt.figure(figsize=(10, 6))
plt.imshow(annotated)
plt.axis("off")
plt.title("VisionGuard Custom PPE Detection")
plt.show()

for box in results[0].boxes:
    cls_name = best_model.names[int(box.cls[0])]
    conf_val = float(box.conf[0])
    print(f"  {cls_name}: {conf_val:.2f}")

## 8. Export for deployment

In [ ]:
# Export to ONNX for deployment on any platform
best_model.export(format="onnx", imgsz=640, dynamic=True)

# Or export to TorchScript for edge devices
# best_model.export(format="torchscript")

print("Model exported. Drop the .onnx file into VisionGuard to use it.")